In [9]:

import requests

parameters = {'key': 'itaaxo7xiphnf6llb0f5l5jbnncgbgro18f4vnl6',
              'place_id': 'colombo',
              'sections': 'current,daily'}

url = "https://www.meteosource.com/api/v1/free/point"

data = requests.get(url, parameters).json()

print(data)
#print('Current temperature in London is {} °C.'.format(data['current']['temperature']))  


In [10]:
#! pip install openmeteo-requests
#! pip install requests-cache retry-requests

In [22]:
import openmeteo_requests

import pandas as pd
import requests_cache
from retry_requests import retry

# Setup the Open-Meteo API client with cache and retry on error
cache_session = requests_cache.CachedSession('.cache', expire_after = -1)
retry_session = retry(cache_session, retries = 5, backoff_factor = 0.2)
openmeteo = openmeteo_requests.Client(session = retry_session)

import datetime
yesterday = (datetime.date.today() - datetime.timedelta(days=1)).isoformat()

# Make sure all required weather variables are listed here
# The order of variables in hourly or daily is important to assign them correctly below
url = "https://archive-api.open-meteo.com/v1/archive"
params = {
    "latitude": 6.9355,
    "longitude": 79.8487,
    "start_date": yesterday,
    "end_date": yesterday,
    "hourly": ["temperature_2m", "precipitation"],
    "daily": ["temperature_2m_min", "temperature_2m_max", "precipitation_sum"],
    "timezone": "auto"
}
responses = openmeteo.weather_api(url, params=params)

# Process first location. Add a for-loop for multiple locations or weather models
response = responses[0]
print(f"Coordinates: {response.Latitude()}°N {response.Longitude()}°E")
print(f"Elevation: {response.Elevation()} m asl")
print(f"Timezone: {response.Timezone()}{response.TimezoneAbbreviation()}")
print(f"Timezone difference to GMT+0: {response.UtcOffsetSeconds()}s")

# Process hourly data. The order of variables needs to be the same as requested.
hourly = response.Hourly()
hourly_temperature_2m = hourly.Variables(0).ValuesAsNumpy()
hourly_precipitation = hourly.Variables(1).ValuesAsNumpy()

hourly_data = {"date": pd.date_range(
	start = pd.to_datetime(hourly.Time(), unit = "s", utc = True),
	end = pd.to_datetime(hourly.TimeEnd(), unit = "s", utc = True),
	freq = pd.Timedelta(seconds = hourly.Interval()),
	inclusive = "left"
)}

hourly_data["temperature_2m"] = hourly_temperature_2m
hourly_data['precipitation'] = hourly_precipitation

hourly_dataframe = pd.DataFrame(data = hourly_data)
print("\nHourly data\n", hourly_dataframe)


# Process daily data (min/max temp and rainfall sum)
daily = response.Daily()

daily_min_temp = daily.Variables(0).ValuesAsNumpy()[0]
daily_max_temp = daily.Variables(1).ValuesAsNumpy()[0]
daily_rainfall = daily.Variables(2).ValuesAsNumpy()[0]

In [23]:
print(f"\nDaily Summary for {params['start_date']}")
print(f"Minimum Temperature: {daily_min_temp:.1f} °C")
print(f"Maximum Temperature: {daily_max_temp:.1f} °C")
average_temp = hourly_dataframe["temperature_2m"].mean()
print(f"Average Temperature: {average_temp:.1f} °C")
print(f"Total Rainfall: {daily_rainfall:.1f} mm")

calculate everything you need from the hourly data

In [21]:
# Minimum temperature
min_temp = hourly_dataframe["temperature_2m"].min()

# Maximum temperature
max_temp = hourly_dataframe["temperature_2m"].max()

# Average temperature
avg_temp = hourly_dataframe["temperature_2m"].mean()

# Total rainfall (sum of hourly precipitation)
total_rainfall = hourly_dataframe["precipitation"].sum()

print(f"Minimum Temperature: {min_temp:.1f} °C")
print(f"Maximum Temperature: {max_temp:.1f} °C")
print(f"Average Temperature: {avg_temp:.1f} °C")
print(f"Total Rainfall: {total_rainfall:.1f} mm")

In [1]:
# !pip install openmeteo-requests requests-cache retry-requests pandas

import pandas as pd
import openmeteo_requests
import requests_cache
from retry_requests import retry

# Set up cached + retry session
cache_session = requests_cache.CachedSession('.cache', expire_after=-1)
retry_session = retry(cache_session, retries=5, backoff_factor=0.25)
openmeteo = openmeteo_requests.Client(session=retry_session)

# Colombo coordinates
LAT, LON = 6.9355, 79.8487

# Request last 2 days so we cover any boundary
params = {
    "latitude": LAT,
    "longitude": LON,
    "start_date": "2025-10-25",     # you can make this dynamic
    "end_date": "2025-10-27",       # include today's date
    "hourly": ["temperature_2m", "precipitation"],
    "timezone": "auto"
}

responses = openmeteo.weather_api("https://archive-api.open-meteo.com/v1/archive", params=params)
response = responses[0]

hourly = response.Hourly()
time_index = pd.date_range(
    start=pd.to_datetime(hourly.Time(), unit="s", utc=True),
    end=pd.to_datetime(hourly.TimeEnd(), unit="s", utc=True),
    freq=pd.Timedelta(seconds=hourly.Interval()),
    inclusive="left"
)

df = pd.DataFrame({
    "time_utc": time_index,
    "temperature_2m": hourly.Variables(0).ValuesAsNumpy(),
    "precipitation": hourly.Variables(1).ValuesAsNumpy()
})
df["time"] = df["time_utc"].dt.tz_convert("Asia/Colombo")

# ✅ Show the very last available record
print("Last available timestamp (Colombo time):", df["time"].max())
print(df.tail())
